In [2]:
import os, time, shutil, glob
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, NoAlertPresentException
from datetime import datetime

# ── CONFIG ─────────────────────────────────────────────────────
CHROMEDRIVER  = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\chromedriver-win64\chromedriver.exe"
SOURCE_FOLDER = r"C:\temp\expedia_downloads"
BASE_CAPTURE  = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE"

DIRS = {
    "current_agent"    : os.path.join(BASE_CAPTURE, "current_agent"),
    "lc_rawdata"       : os.path.join(BASE_CAPTURE, "lc_rawdata_in_console"),
    "current_interval" : os.path.join(BASE_CAPTURE, "current_interval"),
}
for d in DIRS.values(): os.makedirs(d, exist_ok=True)

URL_REALTIME  = "https://console.vap.expedia.com/analytics-console-user-interface/optics/agentRealtime"
URL_BREAKDOWN = "https://console.vap.expedia.com/analytics-console-user-interface/optics/agentBreakdownRealtimeDashboard"

# ── HELPERS ────────────────────────────────────────────────────
def check_and_login(driver, url):
    driver.get(url); time.sleep(10)
    try:
        WebDriverWait(driver,10).until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR,'button[data-testid="console-okta-sign-in"]'))).click()
        print("🔑 Signing in..."); time.sleep(2)
        try:
            WebDriverWait(driver,10).until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR,'label[for="input36"][data-se-for-name="rememberMe"]'))).click(); time.sleep(1)
        except TimeoutException: pass
        try:
            WebDriverWait(driver,10).until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR,'input.button.button-primary[type="submit"][value="Next"]'))).click(); time.sleep(10)
        except TimeoutException: pass
        print("🎉 Login successful!")
        try: driver.switch_to.alert.accept()
        except NoAlertPresentException: pass
        driver.get(url)
    except TimeoutException:
        print("✅ No sign-in required.")

def move_files(keyword, dest_dir):
    moved = 0
    for pat in [f"{SOURCE_FOLDER}\\{keyword}*.csv", f"{SOURCE_FOLDER}\\{keyword}*.xlsx"]:
        for fp in glob.glob(pat):
            if fp.endswith(".crdownload"): continue
            dst = os.path.join(dest_dir, os.path.basename(fp))
            if os.path.exists(dst): os.remove(dst)
            shutil.move(fp, dst)
            print(f"  📁 Moved: {os.path.basename(fp)}")
            moved += 1
    if not moved: print(f"  ⚠️ Không có file '{keyword}*' trong SOURCE_FOLDER")
    return moved

def click_download_csv(driver, wait):
    wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR,"div.uitk-menu-container[aria-hidden='false']")))
    wait.until(EC.element_to_be_clickable((By.XPATH,
        "//div[contains(@class,'uitk-menu-open')][@aria-hidden='false']"
        "//span[text()='Download CSV']/ancestor::button"))).click()
    print("  ✅ Clicked Download CSV"); time.sleep(8)

# ── INIT DRIVER ────────────────────────────────────────────────
chrome_options = Options()
chrome_options.add_argument(r"--user-data-dir=C:/temp/new_chrome_profile")
chrome_options.add_argument(r"--profile-directory=Default")
chrome_options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(CHROMEDRIVER), options=chrome_options)
wait   = WebDriverWait(driver, 15)

print(f"\n{'═'*55}")
print(f"🚀 Download Bot started: {datetime.now().strftime('%d-%b-%Y %H:%M:%S')}")
print(f"{'═'*55}")

# ══ STEP 1: agentRealtime — Logged-In Agents ══════════════════
print("\n[1/3] Logged-In Agents CSV")
check_and_login(driver, URL_REALTIME)
try:
    btn = wait.until(lambda d: d.execute_script("""
        const el=Array.from(document.querySelectorAll('*')).find(e=>
            e.childNodes.length===1&&e.childNodes[0].nodeType===Node.TEXT_NODE&&
            e.textContent.trim()==='Logged-In Agents');
        if(!el)return null;
        let n=el.parentElement;
        while(n&&n!==document.body){
            const b=n.querySelectorAll('button.settingsButton');
            if(b.length===1)return b[0]; n=n.parentElement;}
        return null;"""))
    if btn is None: raise Exception("settingsButton not found")
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});",btn); time.sleep(0.5)
    driver.execute_script("arguments[0].click();",btn)
    click_download_csv(driver, wait)
    move_files("Logged-In Agents", DIRS["current_agent"])
except Exception as e:
    print(f"  ❌ {e}")

# ══ STEP 2: agentRealtime — Assigned Workitem (Connect) ═══════
print("\n[2/3] Assigned Workitem (Connect) CSV")
try:
    #driver.get(URL_REALTIME); time.sleep(5)
    btn2 = wait.until(lambda d: d.execute_script("""
        const el=Array.from(document.querySelectorAll('*')).find(e=>
            e.childNodes.length===1&&e.childNodes[0].nodeType===Node.TEXT_NODE&&
            e.textContent.trim()==='Assigned Workitem (Connect)');
        if(!el)return null;
        let n=el.parentElement;
        while(n&&n!==document.body){
            const b=n.querySelectorAll('button.settingsButton');
            if(b.length===1)return b[0]; n=n.parentElement;}
        return null;"""))
    if btn2 is None: raise Exception("settingsButton not found")
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});",btn2); time.sleep(0.5)
    driver.execute_script("arguments[0].click();",btn2)
    click_download_csv(driver, wait)
    move_files("Assigned Workitem (Connect)", DIRS["lc_rawdata"])
except Exception as e:
    print(f"  ❌ {e}")

# ══ STEP 3: agentBreakdownRealtimeDashboard — Current Interval ═
print("\n[3/3] Current Interval CSV")
try:
    check_and_login(driver, URL_BREAKDOWN)
    btns = wait.until(lambda d: d.find_elements(By.CSS_SELECTOR,"button.settingsButton"))
    if not btns: raise Exception("Not found settingsButton")
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});",btns[0]); time.sleep(0.5)
    driver.execute_script("arguments[0].click();",btns[0])
    click_download_csv(driver, wait)
    move_files("Current Interval", DIRS["current_interval"])
except Exception as e:
    print(f"  ❌ {e}")

# ══ DONE ══════════════════════════════════════════════════════
driver.quit()
print(f"\n{'═'*55}")
print(f"✅ Download Bot finished: {datetime.now().strftime('%d-%b-%Y %H:%M:%S')}")
print(f"{'═'*55}\n")


═══════════════════════════════════════════════════════
🚀 Download Bot started: 31-Jul-2026 10:04:08
═══════════════════════════════════════════════════════

[1/3] Logged-In Agents CSV
✅ No sign-in required.
  ✅ Clicked Download CSV
  📁 Moved: Logged-In Agents-Fri Jul 31 2026 10_04_30 GMT+0700 (Indochina Time).csv

[2/3] Assigned Workitem (Connect) CSV
  ✅ Clicked Download CSV
  📁 Moved: Assigned Workitem (Connect)-Fri Jul 31 2026 10_04_39 GMT+0700 (Indochina Time).csv

[3/3] Current Interval CSV
✅ No sign-in required.
  ✅ Clicked Download CSV
  📁 Moved: Current Interval-Fri Jul 31 2026 10_05_09 GMT+0700 (Indochina Time).csv

═══════════════════════════════════════════════════════
✅ Download Bot finished: 31-Jul-2026 10:05:21
═══════════════════════════════════════════════════════



In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import TimeoutException
import os, time, shutil, glob

CHROMEDRIVER  = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\chromedriver-win64\chromedriver.exe"
SOURCE_FOLDER = r"C:\temp\expedia_downloads"
DST_FILE      = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\EN- UCP.xlsx"

FOLDER_URL = (
    "https://cnxmail-my.sharepoint.com/shared?listurl=https%3A%2F%2Fcnxmail-my%2E"
    "sharepoint%2Ecom%2Fpersonal%2Fahmed_ahmedkamh_concentrix_com%2FDocuments"
    "&id=%2Fpersonal%2Fahmed_ahmedkamh_concentrix_com%2FDocuments"
)

# ── Init driver ───────────────────────────────────────────────
chrome_options = Options()
chrome_options.add_argument(r"--user-data-dir=C:/temp/new_chrome_profile")
chrome_options.add_argument(r"--profile-directory=Default")
chrome_options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(CHROMEDRIVER), options=chrome_options)
wait   = WebDriverWait(driver, 20)

driver.get(FOLDER_URL); time.sleep(8)

# ── Locate file element ───────────────────────────────────────
file_el = wait.until(EC.presence_of_element_located((By.XPATH,
    "//span[contains(text(),'EN-') and contains(text(),'UCP')]"
    " | //span[contains(text(),'EN- UCP')]"
    " | //a[contains(@title,'EN-') and contains(@title,'UCP')]"
)))
print(f"✅ Found: {file_el.text or file_el.get_attribute('title')}")

# Scroll into view via JS
driver.execute_script("arguments[0].scrollIntoView({block:'center'});", file_el)
time.sleep(1)

# Right-click via JS dispatch event (avoids ActionChains offscreen issue)
driver.execute_script("""
    arguments[0].dispatchEvent(new MouseEvent('contextmenu', {
        bubbles: true, cancelable: true,
        view: window, button: 2, buttons: 2
    }));
""", file_el)
time.sleep(2)
print("✅ Right-clicked")

# ── Click Download in context menu ───────────────────────────
try:
    dl = wait.until(EC.element_to_be_clickable((By.XPATH,
        "//*[text()='Download' or @aria-label='Download' "
        "or @data-automationid='download' "
        "or contains(@class,'download')]"
    )))
    driver.execute_script("arguments[0].click();", dl)
    print("✅ Clicked Download"); time.sleep(12)
except TimeoutException:
    print("❌ Download button not found — saving page source for debug")
    with open(r"C:\temp\sp_debug.html", "w", encoding="utf-8") as f:
        f.write(driver.page_source)
    print("📄 Saved: C:\\temp\\sp_debug.html")

# ── Move downloaded file to destination ──────────────────────
all_new = glob.glob(f"{SOURCE_FOLDER}\\*")
print(f"Files in SOURCE_FOLDER: {[os.path.basename(f) for f in all_new]}")

moved = False
for fp in all_new:
    if fp.endswith(".crdownload"): continue
    name = os.path.basename(fp).upper()
    if "UCP" in name or ("EN" in name and ".XLSX" in name):
        if os.path.exists(DST_FILE): os.remove(DST_FILE)
        shutil.move(fp, DST_FILE)
        print(f"✅ Moved → {DST_FILE}"); moved = True

if not moved:
    print(f"⚠️ File not found — SOURCE_FOLDER contents: {[os.path.basename(f) for f in all_new]}")

driver.quit()

✅ Found: EN- UCP.xlsx
✅ Right-clicked
✅ Clicked Download
Files in SOURCE_FOLDER: ['EN- UCP.xlsx']
✅ Moved → C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\EN- UCP.xlsx


In [4]:
import openpyxl
import polars as pl
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

UCP_FILE = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\EN- UCP.xlsx"

TZ_VNT = ZoneInfo("Asia/Ho_Chi_Minh")    # UTC+7
TZ_PST = ZoneInfo("America/Los_Angeles")  # PST/PDT tự động DST

def read_range(wb, sheet_name, header_row=2, data_start=3, data_end=50):
    ws      = wb[sheet_name]
    headers = [str(ws.cell(row=header_row, column=c).value or f"Col_{c}").strip()
               for c in range(7, 11)]
    rows = []
    for row in ws.iter_rows(min_row=data_start, max_row=data_end, min_col=7, max_col=10):
        rows.append([cell.value for cell in row])
    df = pl.DataFrame(rows, schema=headers, orient="row")
    return df.filter(pl.any_horizontal(pl.all().is_not_null()))

def gen_intervals(n_rows):
    today    = datetime.now(TZ_VNT).date()
    base_vnt = datetime(today.year, today.month, today.day, 0, 0, tzinfo=TZ_VNT)
    vnt_list, pst_list = [], []
    for i in range(n_rows):
        vnt = base_vnt + timedelta(minutes=30*i)
        pst = vnt.astimezone(TZ_PST)
        vnt_list.append(vnt.strftime("%H:%M"))
        pst_list.append(pst.strftime("%H:%M"))
    return vnt_list, pst_list

def attach_intervals(df, lob):
    n = len(df)
    vnt_list, pst_list = gen_intervals(n)
    return df.with_columns([
        pl.Series("VNT", vnt_list),
        pl.Series("PST", pst_list),
        pl.lit(lob).alias("LOB"),
    ]).select(["LOB", "VNT", "PST"] + df.columns)

wb    = openpyxl.load_workbook(UCP_FILE, data_only=True)
print(f"Sheets: {wb.sheetnames}")

df_nl = read_range(wb, "NL Chat")
df_lg = read_range(wb, "LG Chat")
print(f"NL Chat raw: {df_nl.shape} | LG Chat raw: {df_lg.shape}")

df_nl = attach_intervals(df_nl, "NL Chat")
df_lg = attach_intervals(df_lg, "LG Chat")

df_ucp = (
    pl.concat([df_lg, df_nl], how="diagonal_relaxed")
    .sort(["LOB", "PST"])
)
print(f"\ndf_ucp: {df_ucp.shape}")
print(df_ucp)

c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\openpyxl\reader\excel.py:237: UserWarning: Unknown extension is not supported and will be removed
  ws_parser.bind_all()
c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\openpyxl\reader\excel.py:237: UserWarning: Conditional Formatting extension is not supported and will be removed
  ws_parser.bind_all()


Sheets: ['1st Aug IC Action Plan', 'Cairo PLS Names', 'Voice LG Names', 'Voice NLG Names', 'Sheet3', 'Sheet1', 'Sheet2', 'Chat LIO Names', 'Chat NLG Names', 'Chat LG Names', 'Sheet4', 'Voice NL (2)', 'Chat LIO', 'Sheet5', 'Real time overage and leakage', 'RCA', 'French', 'Spanish', ' Chat LIO', 'Variance All LOB', 'Cross Skilling Metrix SA%', 'NL Voice', 'NL Chat', 'LG Voice', 'LG Chat', ' Chat LIO ', 'EN PLS NLV', 'Halifax(RBC)-PLS', 'Movement', 'NL Chat (2)', 'Hp French', 'HP IT', 'HP GR', 'HP TR', 'HP DE', 'EN PLS Chat', 'EN PLS LG', 'China-PLS', 'Interval view']
NL Chat raw: (48, 4) | LG Chat raw: (48, 4)

df_ucp: (96, 7)
shape: (96, 7)
┌─────────┬───────┬───────┬───────┬─────────┬─────────┬──────┐
│ LOB     ┆ VNT   ┆ PST   ┆ Cairo ┆ Vietnam ┆ Kolkata ┆ Pune │
│ ---     ┆ ---   ┆ ---   ┆ ---   ┆ ---     ┆ ---     ┆ ---  │
│ str     ┆ str   ┆ str   ┆ f64   ┆ f64     ┆ f64     ┆ i64  │
╞═════════╪═══════╪═══════╪═══════╪═════════╪═════════╪══════╡
│ LG Chat ┆ 14:00 ┆ 00:00 ┆ 0.0   ┆ 